In [1]:
import pandas as pd

orders_df = pd.read_csv("orders.csv")
print(orders_df.head())


   order_id  user_id  restaurant_id  order_date  total_amount  \
0         1     2508            450  18-02-2023        842.97   
1         2     2693            309  18-01-2023        546.68   
2         3     2084            107  15-07-2023        163.93   
3         4      319            224  04-10-2023       1155.97   
4         5     1064            293  25-12-2023       1321.91   

                  restaurant_name  
0               New Foods Chinese  
1  Ruchi Curry House Multicuisine  
2           Spice Kitchen Punjabi  
3          Darbar Kitchen Non-Veg  
4       Royal Eatery South Indian  


In [3]:
users_df = pd.read_json("users.json")
print(users_df.head())


   user_id    name       city membership
0        1  User_1    Chennai    Regular
1        2  User_2       Pune       Gold
2        3  User_3  Bangalore       Gold
3        4  User_4  Bangalore    Regular
4        5  User_5       Pune       Gold


In [11]:
import sqlite3
import pandas as pd

# Step 1: Create/connect to a real SQLite database file
db_file = "food_delivery.db"
conn = sqlite3.connect(db_file)
cursor = conn.cursor()

# Step 2: Read your SQL script
with open("restaurants.sql", "r") as f:
    sql_script = f.read()

# Step 3: Execute the SQL script
cursor.executescript(sql_script)
conn.commit()  # Save changes

# Step 4: Check which tables exist
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print("Tables in DB:\n", tables)

# Step 5: Read the restaurants table
restaurants_df = pd.read_sql("SELECT * FROM restaurants", conn)
print("\nRestaurants Data:\n", restaurants_df)

# Step 6: Close connection
conn.close()


Tables in DB:
           name
0  restaurants

Restaurants Data:
      restaurant_id restaurant_name  cuisine  rating
0                1    Restaurant_1  Chinese     4.8
1                2    Restaurant_2   Indian     4.1
2                3    Restaurant_3  Mexican     4.3
3                4    Restaurant_4  Chinese     4.1
4                5    Restaurant_5  Chinese     4.8
..             ...             ...      ...     ...
495            496  Restaurant_496   Indian     3.1
496            497  Restaurant_497  Mexican     4.4
497            498  Restaurant_498  Chinese     3.9
498            499  Restaurant_499  Mexican     4.9
499            500  Restaurant_500  Chinese     3.2

[500 rows x 4 columns]


In [13]:
orders_users_df = pd.merge(
    orders_df,
    users_df,
    on="user_id",
    how="inner"
)

In [15]:
final_df = pd.merge(
    orders_users_df,
    restaurants_df,
    on="restaurant_id",
    how="inner"
)


In [18]:
print(final_df.head())

   order_id  user_id  restaurant_id  order_date  total_amount  \
0         1     2508            450  18-02-2023        842.97   
1         2     2693            309  18-01-2023        546.68   
2         3     2084            107  15-07-2023        163.93   
3         4      319            224  04-10-2023       1155.97   
4         5     1064            293  25-12-2023       1321.91   

                restaurant_name_x       name       city membership  \
0               New Foods Chinese  User_2508  Hyderabad    Regular   
1  Ruchi Curry House Multicuisine  User_2693       Pune    Regular   
2           Spice Kitchen Punjabi  User_2084    Chennai       Gold   
3          Darbar Kitchen Non-Veg   User_319  Bangalore       Gold   
4       Royal Eatery South Indian  User_1064       Pune    Regular   

  restaurant_name_y  cuisine  rating  
0    Restaurant_450  Mexican     3.2  
1    Restaurant_309   Indian     4.5  
2    Restaurant_107  Mexican     4.0  
3    Restaurant_224  Chinese    

In [22]:
# Assuming orders_df has 'user_id' and 'amount'
import pandas as pd

# Group by user_id and sum order amounts
user_total_orders = orders_df.groupby('user_id')['total_amount'].sum().reset_index()

# Filter users whose total order amount > 1000
high_value_users = user_total_orders[user_total_orders['total_amount'] > 1000]

# Count distinct users
num_users = high_value_users['user_id'].nunique()

print("Distinct users with total orders > ₹1000:", num_users)


Distinct users with total orders > ₹1000: 2544


In [26]:
import pandas as pd

# Define rating bins and labels
bins = [3.0, 3.5, 4.0, 4.5, 5.0]
labels = ['3.0–3.5', '3.6–4.0', '4.1–4.5', '4.6–5.0']

# Create a rating range column
final_df['rating_range'] = pd.cut(final_df['rating'], bins=bins, labels=labels, include_lowest=True)

# Sum revenue by rating range
revenue_by_rating = final_df.groupby('rating_range')['total_amount'].sum()

# Find the rating range with highest revenue
top_rating_range = revenue_by_rating.idxmax()
top_revenue = revenue_by_rating.max()

print("Rating range with highest total revenue:", top_rating_range)
print("Total revenue in that range:", top_revenue)


Rating range with highest total revenue: 4.6–5.0
Total revenue in that range: 2197030.75


C:\Users\Yashvin\AppData\Local\Temp\ipykernel_7228\2150527371.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  revenue_by_rating = final_df.groupby('rating_range')['total_amount'].sum()


In [28]:
# Filter only Gold members
gold_orders = final_df[final_df['membership'] == 'Gold']

# Calculate average order value per city
avg_order_city = gold_orders.groupby('city')['total_amount'].mean()

# Find the city with the highest average order value
top_city = avg_order_city.idxmax()
top_avg_value = avg_order_city.max()

print("City with highest average order value among Gold members:", top_city)
print("Average order value in that city:", top_avg_value)


City with highest average order value among Gold members: Chennai
Average order value in that city: 808.4590800299178


In [32]:
import pandas as pd

# Step 1: Calculate number of distinct restaurants per cuisine
restaurants_per_cuisine = final_df.groupby('cuisine')['restaurant_id'].nunique().reset_index()
restaurants_per_cuisine.rename(columns={'restaurant_id': 'num_restaurants'}, inplace=True)

# Step 2: Calculate total revenue per cuisine
revenue_per_cuisine = final_df.groupby('cuisine')['total_amount'].sum().reset_index()
revenue_per_cuisine.rename(columns={'total_amount': 'total_revenue'}, inplace=True)

# Step 3: Merge both metrics
cuisine_stats = pd.merge(restaurants_per_cuisine, revenue_per_cuisine, on='cuisine')

# Step 4: Find cuisines with low number of restaurants but high revenue
# Sort by num_restaurants ascending and total_revenue descending
cuisine_stats_sorted = cuisine_stats.sort_values(by=['num_restaurants', 'total_revenue'], ascending=[True, False])

print("Cuisines with low restaurant count but high revenue:")
print(cuisine_stats_sorted.head(5))  # Top 5 candidates


Cuisines with low restaurant count but high revenue:
   cuisine  num_restaurants  total_revenue
0  Chinese              120     1930504.65
2  Italian              126     2024203.80
1   Indian              126     1971412.58
3  Mexican              128     2085503.09


In [34]:
# Total number of orders
total_orders = len(final_df)

# Number of orders placed by Gold members
gold_orders = len(final_df[final_df['membership'] == 'Gold'])

# Percentage of orders by Gold members
gold_percentage = round((gold_orders / total_orders) * 100)

print("Percentage of total orders placed by Gold members:", gold_percentage, "%")


Percentage of total orders placed by Gold members: 50 %


In [42]:
# Group by restaurant_name_x
restaurant_stats = final_df.groupby('restaurant_name_x').agg(
    total_orders=('order_id', 'count'),
    avg_order_value=('total_amount', 'mean')  # use total_amount since amount column is total_amount
).reset_index()

# Filter restaurants with less than 20 orders
small_restaurants = restaurant_stats[restaurant_stats['total_orders'] < 20]

# Find the restaurant with highest average order value
top_restaurant = small_restaurants.loc[small_restaurants['avg_order_value'].idxmax()]

print("Restaurant with highest average order value (and <20 orders):")
print("Name:", top_restaurant['restaurant_name_x'])
print("Average Order Value:", top_restaurant['avg_order_value'])
print("Total Orders:", top_restaurant['total_orders'])


Restaurant with highest average order value (and <20 orders):
Name: Hotel Dhaba Multicuisine
Average Order Value: 1040.2223076923076
Total Orders: 13


In [40]:
print(final_df.columns)


Index(['order_id', 'user_id', 'restaurant_id', 'order_date', 'total_amount',
       'restaurant_name_x', 'name', 'city', 'membership', 'restaurant_name_y',
       'cuisine', 'rating', 'rating_range'],
      dtype='object')


In [44]:
top_restaurant['restaurant_name_x']


'Hotel Dhaba Multicuisine'

In [46]:
import pandas as pd

# Step 1: Group by city and cuisine
combo_revenue = final_df.groupby(['city', 'cuisine'])['total_amount'].sum().reset_index()

# Step 2: Find the combination with highest revenue
top_combo = combo_revenue.loc[combo_revenue['total_amount'].idxmax()]

print("Combination contributing highest revenue:")
print("City:", top_combo['city'])
print("Cuisine:", top_combo['cuisine'])
print("Total Revenue:", top_combo['total_amount'])


Combination contributing highest revenue:
City: Bangalore
Cuisine: Mexican
Total Revenue: 571004.61


In [48]:
# Group by membership and cuisine
combo_revenue = final_df.groupby(['membership', 'cuisine'])['total_amount'].sum().reset_index()

# Sort descending by revenue
combo_revenue_sorted = combo_revenue.sort_values(by='total_amount', ascending=False)

print("Revenue by membership + cuisine combination:")
print(combo_revenue_sorted)


Revenue by membership + cuisine combination:
  membership  cuisine  total_amount
7    Regular  Mexican    1072943.30
6    Regular  Italian    1018424.75
3       Gold  Mexican    1012559.79
2       Gold  Italian    1005779.05
5    Regular   Indian     992100.27
1       Gold   Indian     979312.31
0       Gold  Chinese     977713.74
4    Regular  Chinese     952790.91


In [52]:
final_df['order_date'] = pd.to_datetime(final_df['order_date'])
final_df['quarter'] = final_df['order_date'].dt.quarter
revenue_by_quarter = final_df.groupby('quarter')['total_amount'].sum().reset_index()

# Sort descending to see highest revenue
revenue_by_quarter = revenue_by_quarter.sort_values(by='total_amount', ascending=False)

print(revenue_by_quarter)
quarter_mapping = {1: 'Q1 (Jan–Mar)', 2: 'Q2 (Apr–Jun)', 3: 'Q3 (Jul–Sep)', 4: 'Q4 (Oct–Dec)'}
top_quarter = revenue_by_quarter.iloc[0]['quarter']

print("Quarter with highest total revenue:", quarter_mapping[top_quarter])


   quarter  total_amount
2        3    2037385.10
3        4    2018263.66
0        1    2010626.64
1        2    1945348.72
Quarter with highest total revenue: Q3 (Jul–Sep)


C:\Users\Yashvin\AppData\Local\Temp\ipykernel_7228\1856925016.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  final_df['order_date'] = pd.to_datetime(final_df['order_date'])


In [54]:
# Filter orders by Gold members and count
gold_orders_count = final_df[final_df['membership'] == 'Gold'].shape[0]

print("Total orders placed by Gold members:", gold_orders_count)


Total orders placed by Gold members: 4987


In [56]:
# Filter orders placed in Hyderabad
hyderabad_orders = final_df[final_df['city'] == 'Hyderabad']

# Sum total revenue and round to nearest integer
hyderabad_revenue = round(hyderabad_orders['total_amount'].sum())

print("Total revenue from Hyderabad:", hyderabad_revenue)


Total revenue from Hyderabad: 1889367


In [58]:
# Count distinct users
distinct_users = final_df['user_id'].nunique()

print("Number of distinct users who placed at least one order:", distinct_users)


Number of distinct users who placed at least one order: 2883


In [60]:
# Filter orders by Gold members
gold_orders = final_df[final_df['membership'] == 'Gold']

# Calculate average order value
avg_order_value = round(gold_orders['total_amount'].mean(), 2)

print("Average order value for Gold members:", avg_order_value)


Average order value for Gold members: 797.15


In [62]:
# Filter orders for restaurants with rating >= 4.5
high_rating_orders = final_df[final_df['rating'] >= 4.5]

# Count total orders
num_high_rating_orders = high_rating_orders.shape[0]

print("Number of orders for restaurants with rating ≥ 4.5:", num_high_rating_orders)


Number of orders for restaurants with rating ≥ 4.5: 3374


In [64]:
# Step 1: Filter only Gold member orders
gold_orders = final_df[final_df['membership'] == 'Gold']

# Step 2: Find the top revenue city among Gold members
top_city = gold_orders.groupby('city')['total_amount'].sum().idxmax()

# Step 3: Count the number of orders in that city
num_orders_top_city = gold_orders[gold_orders['city'] == top_city].shape[0]

print("Number of orders in top revenue city among Gold members:", num_orders_top_city)


Number of orders in top revenue city among Gold members: 1337


In [66]:
len(final_df)


10000

In [68]:
orders_users_df = pd.merge(orders_df, users_df, on='user_id', how='left')
